# Generative Adversarial Networks (GANs)

**Learning Objectives:** Understand GAN concepts (generator vs discriminator), adversarial training, DCGAN architecture, training challenges, and build a GAN to generate synthetic images.

## 1. GAN Concept

A GAN consists of two networks competing in a **minimax game**:

- **Generator (G):** Takes random noise $z$ and generates fake data $G(z)$ that looks real
- **Discriminator (D):** Distinguishes real data $x$ from fake data $G(z)$

**Generator's goal:** Fool the discriminator
**Discriminator's goal:** Correctly classify real vs fake

**Minimax objective:**
$\min_G \max_D V(D, G) = \mathbb{E}_{x \sim p_{data}}[\log D(x)] + \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]$

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

## 2. Simple GAN for MNIST

We'll build a GAN with dense layers for generating MNIST-like digits.

In [ ]:
# Generator: noise -> image
def build_generator(latent_dim=100):
    model = keras.Sequential([
        layers.Dense(128, input_dim=latent_dim),
        layers.LeakyReLU(alpha=0.01),
        layers.BatchNormalization(),
        layers.Dense(256),
        layers.LeakyReLU(alpha=0.01),
        layers.BatchNormalization(),
        layers.Dense(512),
        layers.LeakyReLU(alpha=0.01),
        layers.BatchNormalization(),
        layers.Dense(784, activation='tanh')  # tanh for [-1, 1] output
    ])
    return model

# Discriminator: image -> real/fake
def build_discriminator():
    model = keras.Sequential([
        layers.Dense(512, input_shape=(784,)),
        layers.LeakyReLU(alpha=0.01),
        layers.Dropout(0.3),
        layers.Dense(256),
        layers.LeakyReLU(alpha=0.01),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

latent_dim = 100
generator = build_generator(latent_dim)
discriminator = build_discriminator()

print("Generator and discriminator built.")

In [ ]:
# Load and preprocess MNIST
(x_train, _), (_, _) = keras.datasets.mnist.load_data()

# Normalize to [-1, 1] (tanh output range)
x_train = (x_train.astype('float32') - 127.5) / 127.5
x_train = x_train.reshape(-1, 784)

batch_size = 128
dataset = tf.data.Dataset.from_tensor_slices(x_train)
dataset = dataset.shuffle(60000).batch(batch_size)

print(f"Data ready: {x_train.shape}")

## 3. Adversarial Training Loop

GANs are trained with alternating updates:
1. Train discriminator on real + fake batches
2. Train generator via discriminator (with generator weights frozen in combined model)

In [ ]:
# Loss functions and optimizers
cross_entropy = keras.losses.BinaryCrossentropy()
g_optimizer = keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)
d_optimizer = keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)

# Discriminator loss
def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss

# Generator loss: want discriminator to classify fakes as real
def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

# Combined GAN model
discriminator.compile(optimizer=d_optimizer, loss='binary_crossentropy')

# Generator is trained through the combined model
discriminator.trainable = False
gan_input = keras.Input(shape=(latent_dim,))
fake_image = generator(gan_input)
gan_output = discriminator(fake_image)
gan = keras.Model(gan_input, gan_output)
gan.compile(optimizer=g_optimizer, loss='binary_crossentropy')

print("GAN combined model ready.")

In [ ]:
# Custom training loop
def train_gan(epochs=50, save_interval=10):
    real = np.ones((batch_size, 1))
    fake = np.zeros((batch_size, 1))

    for epoch in range(epochs):
        for batch in dataset:
            current_batch = batch.shape[0]

            # 1. Train discriminator
            noise = np.random.normal(0, 1, (current_batch, latent_dim))
            generated_images = generator.predict(noise, verbose=0)

            real_labels = np.ones((current_batch, 1)) * 0.9  # label smoothing
            fake_labels = np.zeros((current_batch, 1))

            d_loss_real = discriminator.train_on_batch(batch, real_labels)
            d_loss_fake = discriminator.train_on_batch(generated_images, fake_labels)
            d_loss = 0.5 * (d_loss_real + d_loss_fake)

            # 2. Train generator
            noise = np.random.normal(0, 1, (current_batch, latent_dim))
            misleading_labels = np.ones((current_batch, 1))
            g_loss = gan.train_on_batch(noise, misleading_labels)

        if epoch % save_interval == 0:
            print(f"Epoch {epoch}: D loss = {d_loss:.4f}, G loss = {g_loss:.4f}")
            generate_and_save_images(epoch)

def generate_and_save_images(epoch):
    noise = np.random.normal(0, 1, (25, latent_dim))
    generated = generator.predict(noise, verbose=0)
    generated = (generated + 1) / 2  # rescale to [0, 1]

    fig, axes = plt.subplots(5, 5, figsize=(8, 8))
    for i, ax in enumerate(axes.flat):
        ax.imshow(generated[i].reshape(28, 28), cmap='gray')
        ax.axis('off')
    plt.suptitle(f'Generated Images - Epoch {epoch}')
    plt.show()

print("Training function defined. Call train_gan() to start.")

In [ ]:
# Train the GAN (short run for demo)
train_gan(epochs=20, save_interval=5)

## 4. DCGAN — Deep Convolutional GAN

DCGAN uses convolutional layers instead of dense layers for more stable training and higher quality images.

**DCGAN guidelines:**
- Use Conv2DTranspose for upsampling in generator
- Use strided convolutions for downsampling in discriminator
- Use BatchNormalization (except generator output, discriminator input)
- Use LeakyReLU in discriminator (alpha=0.2)
- Use ReLU in generator (except output which uses tanh)
- Use Adam optimizer (LR=0.0002, beta_1=0.5)

In [ ]:
# DCGAN generator
def build_dcgan_generator(latent_dim=100):
    model = keras.Sequential([
        layers.Dense(7*7*256, input_dim=latent_dim),
        layers.Reshape((7, 7, 256)),
        layers.BatchNormalization(),
        layers.ReLU(),

        # 7x7 -> 14x14
        layers.Conv2DTranspose(128, (5, 5), strides=2, padding='same'),
        layers.BatchNormalization(),
        layers.ReLU(),

        # 14x14 -> 28x28
        layers.Conv2DTranspose(64, (5, 5), strides=2, padding='same'),
        layers.BatchNormalization(),
        layers.ReLU(),

        # 28x28 -> 28x28 (final)
        layers.Conv2D(1, (5, 5), padding='same', activation='tanh')
    ])
    return model

# DCGAN discriminator
def build_dcgan_discriminator(img_shape=(28, 28, 1)):
    model = keras.Sequential([
        layers.Conv2D(64, (5, 5), strides=2, padding='same',
                      input_shape=img_shape),
        layers.LeakyReLU(alpha=0.2),
        layers.Dropout(0.3),

        layers.Conv2D(128, (5, 5), strides=2, padding='same'),
        layers.LeakyReLU(alpha=0.2),
        layers.Dropout(0.3),

        layers.Flatten(),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

dcgan_gen = build_dcgan_generator()
dcgan_disc = build_dcgan_discriminator()
print("DCGAN models built.")

## 5. Training Challenges

| Challenge | Description | Solution |
|-----------|-------------|----------|
| **Mode collapse** | Generator produces limited variety | Minibatch discrimination, unrolled GANs |
| **Non-convergence** | Loss oscillates, never stabilizes | Wasserstein loss, gradient penalty |
| **Vanishing gradients** | Discriminator too strong | Label smoothing, instance noise |
| **Unstable training** | Both networks diverge | Batch norm, careful LR tuning |

**Practical tips:**
- Use label smoothing (real labels = 0.9 instead of 1.0)
- Train discriminator less often than generator
- Add noise to discriminator inputs
- Use Wasserstein loss (WGAN) for stability

In [ ]:
# Label smoothing demonstration
def smooth_labels(labels, smoothing=0.1):
    return labels * (1 - smoothing) + smoothing * 0.5

print("Label smoothing example:")
print(f"Real label: 1.0 -> {smooth_labels(np.array([1.0]))[0]:.4f}")
print(f"Fake label: 0.0 -> {smooth_labels(np.array([0.0]))[0]:.4f}")

# Conditional GAN concept (brief)
print("\nConditional GAN (CGAN): Provide class labels to both G and D for class-conditional generation.")

## 6. Interview Problems

### Problem 1: Build a GAN to Generate MNIST Digits

Implement a GAN with dense layers that generates recognizable MNIST digits. Show generated images at epochs 1, 10, 50. Discuss the quality evolution.

### Problem 2: Improve GAN Training with Label Smoothing

Compare GAN training with and without label smoothing. Show that smoothing reduces discriminator confidence and leads to more stable training.

### Problem 3: DCGAN for CIFAR-10

Build a DCGAN to generate CIFAR-10 images (32×32×3). Use Conv2DTranspose in generator and strided convolutions in discriminator.

In [ ]:
# Solution: Problem 2 — Label smoothing comparison
def train_gan_with_label_smoothing(use_smoothing=False, epochs=10):
    # Reset models
    g = build_generator()
    d = build_discriminator()
    d.compile(optimizer=d_optimizer, loss='binary_crossentropy')
    d.trainable = False
    combined = keras.Sequential([g, d])
    combined.compile(optimizer=g_optimizer, loss='binary_crossentropy')

    d_losses = []
    g_losses = []

    for epoch in range(epochs):
        for batch in dataset.take(30):  # subset for speed
            current_batch = batch.shape[0]
            noise = np.random.normal(0, 1, (current_batch, latent_dim))
            gen_imgs = g.predict(noise, verbose=0)

            if use_smoothing:
                real_labels = np.ones((current_batch, 1)) * 0.9
            else:
                real_labels = np.ones((current_batch, 1))
            fake_labels = np.zeros((current_batch, 1))

            d_loss = 0.5 * (d.train_on_batch(batch, real_labels) +
                            d.train_on_batch(gen_imgs, fake_labels))
            noise = np.random.normal(0, 1, (current_batch, latent_dim))
            g_loss = combined.train_on_batch(noise, np.ones((current_batch, 1)))

        d_losses.append(d_loss)
        g_losses.append(g_loss)

    return d_losses, g_losses

print("Comparison function defined.")

## Summary

- **GANs** consist of a generator (creates fake data) and discriminator (distinguishes real vs fake)
- **Minimax game** balances the two networks during adversarial training
- **DCGAN** uses convolutional layers for higher-quality image generation
- **Training challenges:** Mode collapse, non-convergence, unstable gradients
- **Improvements:** Label smoothing, batch normalization, LeakyReLU, Adam (beta_1=0.5)
- **Conditional GANs** (CGAN) add class labels for controlled generation
- Applications: image generation, super-resolution, style transfer, data augmentation